# B-field visualization

Loads a snapshot, converts the magnetic field to µG, projects it onto the XY plane (density-weighted), and plots field strength with orientation bars. The snapshot needs to be from an MHD run — if there's no `MagneticField` block you'll get an error early.

Feel free to reach out if anything isn't clear!

In [ ]:
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path("..").resolve()))

from simviz.utils import read_snapshot_hdf5, calc_bfield_uG, CONSTANTS
from simviz.projections import rotate_to_bar_frame
from simviz.field_plots import project_bfield_xy, plot_bfield_xy

EXAMPLE_OUT = Path("../example_output")
EXAMPLE_OUT.mkdir(exist_ok=True)

# Cache file for the bar-frame-rotated arrays (avoids reloading the snapshot every run).
BFIELD_CACHE = EXAMPLE_OUT / ".bfield_cache.npz"

Point `snap_path` at your snapshot file and run from here.

In [ ]:
snap_path = Path("../sample_snaps/phoenix_stinks_1Msun_999.hdf5")

data, header = read_snapshot_hdf5(snap_path)

t   = float(header["Time"])
box = float(header["BoxSize"])
print(f"t = {t:.4f} code  ({t * 98.7:.1f} Myr)   BoxSize = {box}")

if "MagneticField" not in data:
    raise RuntimeError("Snapshot has no MagneticField — was MHD enabled for this run?")

Convert to µG and rotate into the bar frame. The B components need to rotate by the same angle as the positions-`rotate_to_bar_frame` handles both at once, just pass `Bx, By` where you'd normally put velocities.

In [ ]:
x, y, z    = np.array(data["Coordinates"]).T
Bx, By, Bz = calc_bfield_uG(data["MagneticField"]).T
density    = np.array(data["Density"])

# Centre on GC
x -= box / 2
y -= box / 2
z -= box / 2

# Rotate positions and B vector components into the bar frame together
x, y, Bx, By = rotate_to_bar_frame(x, y, Bx, By, t)

B_total = np.sqrt(Bx**2 + By**2 + Bz**2)
print(f"B field range: {B_total.min():.2e} – {B_total.max():.2e} µG   mean: {B_total.mean():.2e} µG")

Loading and rotating snapshots be slo. Run the save cell once, then on future runs you can skip straight to loading and go from there.

In [ ]:
# SAVE — run once after sections 1–2
np.savez(BFIELD_CACHE, x=x, y=y, z=z, Bx=Bx, By=By, Bz=Bz, density=density, t=np.array(t))
print("Saved →", BFIELD_CACHE)

In [ ]:
# LOAD — run instead of sections 1–2 to restore from cache
_d = np.load(BFIELD_CACHE)
x, y, z, Bx, By, Bz = _d["x"], _d["y"], _d["z"], _d["Bx"], _d["By"], _d["Bz"]
density = _d["density"]
t = float(_d["t"])
print("Loaded ←", BFIELD_CACHE)

Project onto a 2D XY grid using density-weighted binning. This is much faster than the old `NearestNDInterpolator` approach — if you want a different spatial range or resolution just pass a `grid=` dict.

In [ ]:
coords  = np.column_stack([x, y, z])
bvecs   = np.column_stack([Bx, By, Bz])

Bx_proj, By_proj, Bz_proj, B_mag, xs, ys = project_bfield_xy(coords, bvecs, density)

print(f"Grid shape: {B_mag.shape}   B_mag range: {B_mag[B_mag>0].min():.2e} – {B_mag.max():.2e} µG")

The white bars show projected field orientation — they're headless on purpose since we can't infer direction from projected B. Adjust `vmin`/`vmax` and `orientation_stride` to taste.

In [ ]:
plt.style.use("dark_background")

fig, ax = plot_bfield_xy(
    B_mag, Bx_proj, By_proj, xs, ys,
    vmin=1e0, vmax=1e3,
    show_orientation=True,
    orientation_stride=30,
)
ax.set_title(f"t = {t * 98.7:.1f} Myr", loc="left")
fig.savefig(EXAMPLE_OUT / "bfield_xy_main.png", dpi=180, bbox_inches="tight")
plt.show()

Zoom into the inner CMZ
 just change the `custom_grid` bounds and re-project

In [ ]:
# Zoom to inner CMZ region, coarser orientation grid
if "project_bfield_xy" not in globals() or "plot_bfield_xy" not in globals():
    from simviz.field_plots import project_bfield_xy, plot_bfield_xy

custom_grid = {
    "xmin": -1.5,
    "xmax": 1.5,
    "dx": 0.01,
    "ymin": -1.5,
    "ymax": 1.5,
    "dy": 0.01,
}
# Higher-res option (slower): set dx=dy=0.005

Bx2, By2, Bz2, B2, xs2, ys2 = project_bfield_xy(coords, bvecs, density, grid=custom_grid)

fig2, ax2 = plot_bfield_xy(
    B2,
    Bx2,
    By2,
    xs2,
    ys2,
    vmin=1e0,
    vmax=1e3,
    show_orientation=True,
    orientation_stride=24,
)
ax2.set_title("inner CMZ", loc="left")
fig2.savefig(EXAMPLE_OUT / "bfield_xy_inner_cmz.png", dpi=180, bbox_inches="tight")
plt.show()